# POLAPP KCDD 위협 문맥 분류 모델

공식 KCDD train/dev/test 분할과 `klue/roberta-small`을 사용합니다. 논문 보고용 최종 실행은 `SEEDS = [13, 42, 77]`을 유지하고, 연결 확인만 할 때는 `[42]`로 줄일 수 있습니다. Colab 런타임은 GPU로 설정하세요.

In [ ]:
!pip -q install 'transformers>=4.46,<5' 'datasets>=3,<5' 'accelerate>=1,<2' 'optimum[onnxruntime]>=1.23,<2' 'onnxruntime>=1.19' 'scikit-learn>=1.4'

In [ ]:
import json, shutil, statistics, urllib.request, zipfile
from pathlib import Path
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

MODEL_NAME = 'klue/roberta-small'
SEEDS = [13, 42, 77]
EPOCHS = 4
MAX_LENGTH = 256
ROOT = Path('/content/polapp_kcdd')
ROOT.mkdir(exist_ok=True)
LABEL_CODES = ['000001', '020121', '02051', '020811', '020819']
LABEL_NAMES = ['Clean Dialogue', 'Serious Threats', 'Extortion or Blackmail', 'Harassment in Workplace', 'Other Harassment']
code_to_id = {code: index for index, code in enumerate(LABEL_CODES)}

In [ ]:
# EACL 2024 Findings 공식 software attachment에 데이터와 원 논문 코드가 포함되어 있습니다.
archive = ROOT / 'kcdd_software.zip'
url = 'https://aclanthology.org/attachments/2024.findings-eacl.42.software.zip'
urllib.request.urlretrieve(url, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(ROOT / 'official')
data_root = next(path for path in (ROOT / 'official').rglob('data') if (path / 'train.txt').exists())
print('Official data:', data_root)

In [ ]:
def read_split(path):
    texts, labels = [], []
    with Path(path).open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 2 or parts[1] not in code_to_id:
                raise ValueError(f'Unexpected KCDD row at {path}:{line_number}')
            texts.append(parts[0].strip())
            labels.append(code_to_id[parts[1]])
    return Dataset.from_dict({'text': texts, 'label': labels})

datasets = DatasetDict({
    'train': read_split(data_root / 'train.txt'),
    'validation': read_split(data_root / 'dev.txt'),
    'test': read_split(data_root / 'test.txt'),
})
print(datasets)
for split in datasets:
    values, counts = np.unique(datasets[split]['label'], return_counts=True)
    print(split, {LABEL_NAMES[int(v)]: int(c) for v, c in zip(values, counts)})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
tokenized = datasets.map(tokenize, batched=True, remove_columns=['text'])
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def metrics_from_logits(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    precision, recall, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'macro_precision': precision,
        'macro_recall': recall,
        'macro_f1': macro_f1,
    }

In [ ]:
id2label = {index: name for index, name in enumerate(LABEL_NAMES)}
label2id = {name: index for index, name in id2label.items()}
run_metrics, final_dirs = [], []

for seed in SEEDS:
    set_seed(seed)
    run_dir = ROOT / f'seed_{seed}'
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABEL_NAMES), id2label=id2label, label2id=label2id
    )
    args = TrainingArguments(
        output_dir=str(run_dir / 'checkpoints'),
        learning_rate=5e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='steps',
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        save_total_limit=1,
        fp16=True,
        seed=seed, data_seed=seed,
        report_to='none',
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'], processing_class=tokenizer,
        data_collator=collator, compute_metrics=metrics_from_logits,
    )
    trainer.train()
    final_dir = run_dir / 'final'
    trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)
    prediction = trainer.predict(tokenized['test'])
    predicted = np.argmax(prediction.predictions, axis=-1)
    result = {'seed': seed, **{k.replace('test_', ''): float(v) for k, v in prediction.metrics.items() if isinstance(v, (int, float))}}
    result['classification_report'] = classification_report(prediction.label_ids, predicted, target_names=LABEL_NAMES, output_dict=True, zero_division=0)
    result['confusion_matrix'] = confusion_matrix(prediction.label_ids, predicted).tolist()
    run_metrics.append(result)
    final_dirs.append(final_dir)
    print(seed, result['macro_f1'], result['accuracy'])

In [ ]:
best_index = int(np.argmax([item['macro_f1'] for item in run_metrics]))
best_dir = final_dirs[best_index]
summary = {
    'dataset': 'KCDD official train/dev/test split',
    'base_model': MODEL_NAME,
    'seeds': SEEDS,
    'runs': run_metrics,
    'test_macro_f1_mean': statistics.mean(item['macro_f1'] for item in run_metrics),
    'test_macro_f1_std': statistics.pstdev(item['macro_f1'] for item in run_metrics),
    'test_accuracy_mean': statistics.mean(item['accuracy'] for item in run_metrics),
    'test_accuracy_std': statistics.pstdev(item['accuracy'] for item in run_metrics),
    'selected_seed': SEEDS[best_index],
}
print(json.dumps({k: v for k, v in summary.items() if k != 'runs'}, indent=2))

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from onnxruntime.quantization import QuantType, quantize_dynamic

export_dir = ROOT / 'guardian_context_model'
ort_model = ORTModelForSequenceClassification.from_pretrained(best_dir, export=True)
ort_model.save_pretrained(export_dir)
tokenizer.save_pretrained(export_dir)
onnx_path = next(export_dir.glob('*.onnx'))
int8_path = export_dir / 'model.int8.onnx'
quantize_dynamic(str(onnx_path), str(int8_path), weight_type=QuantType.QInt8)
if onnx_path != int8_path:
    onnx_path.unlink()

(export_dir / 'labels.json').write_text(json.dumps({'id2label': id2label, 'label_codes': LABEL_CODES}, indent=2), encoding='utf-8')
(export_dir / 'thresholds.json').write_text(json.dumps({'context_threat_threshold': 0.50, 'note': 'calibrate on POLAPP field validation set'}, indent=2), encoding='utf-8')
(export_dir / 'metrics.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
card = f'''# POLAPP Guardian Context Model

Base: {MODEL_NAME}
Dataset: KCDD official split
Seeds: {SEEDS}
Test Macro-F1 mean/std: {summary['test_macro_f1_mean']:.4f} / {summary['test_macro_f1_std']:.4f}
Test Accuracy mean/std: {summary['test_accuracy_mean']:.4f} / {summary['test_accuracy_std']:.4f}

Limitations: KCDD is an online-dialogue corpus, so field audio transcripts require separate external validation. Output is decision support, not an automatic police action.
'''
(export_dir / 'model_card.md').write_text(card, encoding='utf-8')
print(list(export_dir.iterdir()))

In [ ]:
# INT8 모델이 실제로 열리고 추론되는지 확인한 뒤 압축합니다.
import onnxruntime as ort
session = ort.InferenceSession(str(export_dir / 'model.int8.onnx'), providers=['CPUExecutionProvider'])
sample = tokenizer('죽여 버리겠다', return_tensors='np', truncation=True, padding='max_length', max_length=MAX_LENGTH)
feed = {item.name: sample[item.name].astype(np.int64) for item in session.get_inputs()}
logits = session.run(None, feed)[0]
print('ONNX logits:', logits)
archive_path = shutil.make_archive('/content/guardian_context_model', 'zip', export_dir)
from google.colab import files
files.download(archive_path)